## 06.MODEL EVALUATION

### Objective
The objective of this section is to evaluate the performance of the five trained machine learning classification models using different evaluation metrics.

In [1]:
# Import libraries for data handling and evaluation
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Import the five classification algorithms
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [2]:
# Load the student performance dataset
df = pd.read_csv("../data/student-mat.csv", sep=";")

# Create the Pass/Fail target
df["Result"] = df["G3"].apply(
    lambda x: "Pass" if x >= 10 else "Fail"
)

# Separate input features and target
X = df.drop(columns=["G1", "G2", "G3", "Result"])
y = df["Result"]

print("Input data shape:", X.shape)
print("Target data shape:", y.shape)

Input data shape: (395, 30)
Target data shape: (395,)


In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (316, 30)
Testing data: (79, 30)


### 6.1 Creating and Training the Models

Create the preprocessing transformer and train all five classification models using the training dataset.

In [5]:
# Identify numerical and categorical features
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object","str"]
).columns

# Create a preprocessing function
def create_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
        ]
    )

In [6]:
# Define the five classification models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),

    "SVM": SVC(kernel="rbf", random_state=42),

    "KNN": KNeighborsClassifier(n_neighbors=5),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=1000,
        random_state=42
    )
}

# Train each model using a preprocessing pipeline
trained_models = {}

for name, algorithm in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessing", create_preprocessor()),
            ("model", algorithm)
        ]
    )

    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline

    print(name, "training completed!")

Decision Tree training completed!
Random Forest training completed!
SVM training completed!
KNN training completed!
MLP training completed!


### 6.2 Calculating Evaluation Metrics

Evaluate each model using Accuracy, Precision, Recall, F1-score and ROC-AUC.

In [7]:
# Create an empty list to store evaluation results
results = []

# Convert actual Pass/Fail values into binary values for ROC-AUC
y_test_binary = (y_test == "Pass").astype(int)

# Evaluate each trained model
for name, model in trained_models.items():

    # Predict the results
    predictions = model.predict(X_test)

    # Calculate ROC-AUC scores
    if name == "SVM":
        scores = model.decision_function(X_test)
    else:
        scores = model.predict_proba(X_test)[:, 1]

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test, predictions, pos_label="Pass", zero_division=0
    )

    recall = recall_score(
        y_test, predictions, pos_label="Pass", zero_division=0
    )

    f1 = f1_score(
        y_test, predictions, pos_label="Pass", zero_division=0
    )

    roc_auc = roc_auc_score(y_test_binary, scores)

    # Store the results
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc
    })

# Create a comparison table
results_df = pd.DataFrame(results)

print("Model Performance Comparison:")
display(results_df.round(4))

Model Performance Comparison:


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Decision Tree,0.6076,0.7292,0.6604,0.6931,0.5802
1,Random Forest,0.6582,0.6970,0.8679,0.7731,0.6408
2,SVM,0.6835,0.7059,0.9057,0.7934,0.5980
3,KNN,0.5823,0.6613,0.7736,0.7130,0.5221
4,MLP,0.6582,0.7167,0.8113,0.7611,0.5740


## Confusion Matrix

A confusion matrix shows the number of correct and incorrect Pass/Fail predictions made by each model.

In [8]:
# Display the confusion matrix for each model
for name, model in trained_models.items():

    predictions = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=["Fail", "Pass"]
    )

    print("\nConfusion Matrix for", name)
    print(cm)


Confusion Matrix for Decision Tree
[[13 13]
 [18 35]]

Confusion Matrix for Random Forest
[[ 6 20]
 [ 7 46]]

Confusion Matrix for SVM
[[ 6 20]
 [ 5 48]]

Confusion Matrix for KNN
[[ 5 21]
 [12 41]]

Confusion Matrix for MLP
[[ 9 17]
 [10 43]]


### 6.3 Classification Report

Generate a detailed classification report for each model, including precision, recall and F1-score.

In [9]:
# Display classification reports
for name, model in trained_models.items():

    predictions = model.predict(X_test)

    print("\nClassification Report for", name)

    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )


Classification Report for Decision Tree
              precision    recall  f1-score   support

        Fail       0.42      0.50      0.46        26
        Pass       0.73      0.66      0.69        53

    accuracy                           0.61        79
   macro avg       0.57      0.58      0.57        79
weighted avg       0.63      0.61      0.62        79


Classification Report for Random Forest
              precision    recall  f1-score   support

        Fail       0.46      0.23      0.31        26
        Pass       0.70      0.87      0.77        53

    accuracy                           0.66        79
   macro avg       0.58      0.55      0.54        79
weighted avg       0.62      0.66      0.62        79


Classification Report for SVM
              precision    recall  f1-score   support

        Fail       0.55      0.23      0.32        26
        Pass       0.71      0.91      0.79        53

    accuracy                           0.68        79
   macro avg   

### Conclusion
The evaluation results show that SVM achieved the highest accuracy among the five models. However, the performance of all models in predicting Fail students needs improvement. Therefore, further cross-validation and hyperparameter tuning will be performed to improve model performance and identify the most suitable model for predicting student academic performance.